# 03 - Split and Organize Dataset (70/15/15 stratified)

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../../src')
from yolov8_tumor_detection.data_utils import validate_yolo_label

raw = Path('../../Data/Raw')
# rebuild valid pairs
images = []
labels = []
for p in raw.rglob('*'):
    if p.is_file():
        ext = p.suffix.lower()
        if ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            images.append(p)
        elif ext == '.txt':
            labels.append(p)
label_map = {l.stem: l for l in labels}
img_map = {i.stem: i for i in images}
rows = []
for stem, img in img_map.items():
    if stem not in label_map:
        continue
    lab = label_map[stem]
    v = validate_yolo_label(lab, nc=4)
    if v is None:
        continue
    if len(v) == 0:
        cls = 2  # No Tumor
    else:
        cls = v[0][0]
    rows.append({'image_path': str(img), 'label_path': str(lab), 'cls': cls})
df = pd.DataFrame(rows)
print(len(df))
df['cls'].value_counts()

In [ ]:
# split 70/15/15 stratified
seed = 42
train_df, temp = train_test_split(
    df, test_size=0.3, stratify=df['cls'], random_state=seed
)
val_df, test_df = train_test_split(
    temp, test_size=0.5, stratify=temp['cls'], random_state=seed
)
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'
all_df = pd.concat([train_df, val_df, test_df])
print(len(train_df), len(val_df), len(test_df))
all_df.groupby('split')['cls'].value_counts()

In [ ]:
# copy
import shutil
out = Path('../../Data/processed')
for _, row in all_df.iterrows():
    split = row['split']
    image_source = Path(row['image_path'])
    label_source = Path(row['label_path'])
    image_destination = out / 'images' / split / image_source.name
    label_destination = out / 'labels' / split / label_source.name
    image_destination.parent.mkdir(parents=True, exist_ok=True)
    label_destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_source, image_destination)
    shutil.copy2(label_source, label_destination)
print('done')

In [ ]:
# create data.yaml
names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
yaml = f'''path: ./Data/processed
train: images/train
val: images/val
test: images/test
nc: 4
names: {names}
'''
with open('../../data.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml)
print(yaml)